# 01 — Baseline GSM8K Evaluation

**Notebook 1 of 7.** Evaluates the untouched `Llama-3.2-3B-Instruct` on the first 500 problems of GSM8K test. This is the reference every DPO condition is compared against.

- **Runtime:** ~1.5 hours on a Colab free-tier T4.
- **Output:** `results/baseline_gsm8k_500.json` on Google Drive.
- **Resumable:** if Colab drops mid-run, just re-run the eval cell — it picks up from the first unfinished problem.

Before you start: **Runtime → Change runtime type → T4 GPU**.

## 1. Install dependencies

Unsloth pulls in compatible pinned versions of `trl`, `transformers`, `peft`, `datasets`, and `bitsandbytes`.

In [ ]:
%%capture
!pip install -q unsloth
!pip install -q --force-reinstall --no-cache-dir --no-deps git+https://github.com/unslothai/unsloth.git

## 2. Mount Google Drive and set project paths

Everything the project produces — evaluation JSONs, generated preference data, cached preference sets — lives under `PROJECT_ROOT` on Drive so it survives Colab session drops.

Edit `PROJECT_ROOT` if you want the repo somewhere else on your Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_ROOT = '/content/drive/MyDrive/dpo-reasoning-transfer'
os.makedirs(f'{PROJECT_ROOT}/results', exist_ok=True)
os.makedirs(f'{PROJECT_ROOT}/data', exist_ok=True)
os.makedirs(f'{PROJECT_ROOT}/src', exist_ok=True)
print(f'Project root: {PROJECT_ROOT}')

## 3. Load the shared eval module

Two setup modes:

- **Preferred (once your repo is on GitHub):** uncomment the `git clone` line and point it at your repo. Every session starts by pulling the latest `src/`.
- **Bootstrap (right now):** upload `src/eval.py` and `src/__init__.py` into `PROJECT_ROOT/src/` on Drive manually (drag-and-drop in the Files panel), then just run the cell below.

In [ ]:
# Option A — clone from GitHub (recommended once you've pushed the repo)
# !cd {PROJECT_ROOT} && (git clone https://github.com/YOUR_USERNAME/dpo-reasoning-transfer.git repo \
#   || (cd repo && git pull))
# CODE_DIR = f'{PROJECT_ROOT}/repo'

# Option B — use src/ uploaded directly to Drive (bootstrap mode)
CODE_DIR = PROJECT_ROOT

import sys
if CODE_DIR not in sys.path:
    sys.path.insert(0, CODE_DIR)

try:
    from src.eval import evaluate_gsm8k, extract_answer, answers_match
    print('src/eval.py loaded')
except ImportError as e:
    print(f'Could not import src.eval: {e}')
    print(f'Make sure src/eval.py and src/__init__.py exist under: {CODE_DIR}/src/')
    raise

## 4. Load the base model

`unsloth/Llama-3.2-3B-Instruct` in 4-bit. First-run download is ~2 GB; loading takes 2–3 minutes.

In [ ]:
import torch
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = 'unsloth/Llama-3.2-3B-Instruct',
    max_seq_length = 2048,
    dtype = None,          # auto
    load_in_4bit = True,
)
print(f'Model device: {next(model.parameters()).device}')
print(f'CUDA available: {torch.cuda.is_available()}')

## 5. (Optional) Keepalive

Colab disconnects idle browser tabs after ~90 minutes. This snippet nudges the tab every minute so a long eval finishes even if you step away. Skip if you're going to watch the run.

In [ ]:
from IPython.display import display, Javascript
display(Javascript('''
function _keepAlive() {
    let btn = document.querySelector('colab-toolbar-button#connect');
    if (btn) { btn.click(); }
}
setInterval(_keepAlive, 60000);
'''))
print('Keepalive registered.')

## 6. Run baseline evaluation

Calls the shared `evaluate_gsm8k` from `src/eval.py`. Progress bar shows live accuracy. Per-problem records are appended to `.partial.jsonl` as we go; if Colab drops, just re-run this cell.

Expected runtime on T4: ~1.5 hours for 500 problems.

In [ ]:
results = evaluate_gsm8k(
    model=model,
    tokenizer=tokenizer,
    n_samples=500,
    output_path=f'{PROJECT_ROOT}/results/baseline_gsm8k_500.json',
    max_new_tokens=512,
    run_metadata={
        'condition': 'baseline',
        'base_model': 'unsloth/Llama-3.2-3B-Instruct',
        'quantization': '4-bit',
        'notebook': '01_baseline_eval',
    },
)
print()
print('=' * 60)
print(f"Baseline accuracy: {results['accuracy']:.1%} "
      f"({results['n_correct']}/{results['n_samples']})")
print('=' * 60)

## 7. Sanity check

Two correct and two wrong predictions, so you can eyeball that extraction is working and the model is doing what you expect.

In [ ]:
correct_examples = [r for r in results['results'] if r['correct']][:2]
wrong_examples   = [r for r in results['results'] if not r['correct']][:2]

for label, examples in [('CORRECT', correct_examples), ('WRONG', wrong_examples)]:
    print(f"\n{'=' * 60}\n{label}\n{'=' * 60}")
    for r in examples:
        print(f"\nQ (first 140 chars): {r['question'][:140]}...")
        print(f"GT: {r['gt_answer']} | Pred: {r['pred_answer']}")
        print(f"Response tail: ...{r['response'][-160:]}")

## 8. Basic diagnostics

Response-length distribution — useful reference for later notebooks so we can spot when DPO shifts verbosity.

In [ ]:
import statistics

lens_chars = [len(r['response']) for r in results['results']]
lens_words = [len(r['response'].split()) for r in results['results']]

print(f"Response length (chars): mean={statistics.mean(lens_chars):.0f}, "
      f"median={statistics.median(lens_chars):.0f}, "
      f"sd={statistics.stdev(lens_chars):.0f}")
print(f"Response length (words): mean={statistics.mean(lens_words):.0f}, "
      f"median={statistics.median(lens_words):.0f}, "
      f"sd={statistics.stdev(lens_words):.0f}")

# Sanity: how often did the extractor fail?
none_preds = sum(1 for r in results['results'] if r['pred_answer'] is None)
print(f"\nProblems where answer extraction returned None: {none_preds}/{len(results['results'])}")

## Next

- `results/baseline_gsm8k_500.json` on your Drive is the anchor every DPO condition is compared against. **Do not overwrite it.**
- **Next up:** `02_build_reasoning_prefs.ipynb` — sample chains-of-thought from this same base model on the GSM8K *train* split, label them correct/incorrect, and build the 500-pair Condition B preference set.

If you want to spot-check the cached JSON structure:
```python
import json
with open(f'{PROJECT_ROOT}/results/baseline_gsm8k_500.json') as f:
    cached = json.load(f)
print(list(cached.keys()))
print(cached['accuracy'], cached['n_correct'], '/', cached['n_samples'])
```